# Token Model — Baseline + early stopping based on validation loss, label smoothing, warmup + cosine, mixed precision training

In [4]:
%tb
import os, json, math, random, glob
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

import warnings
warnings.filterwarnings("ignore")

from modules.plotting import MetricLog, plot_metrics
from modules.hand_testing import hand_test_repl
from modules.best_model_saver import BestModelSaver
from modules.early_stopping import EarlyStopping
from modules.datasets.loading import load_files
from modules.datasets.token_dataset import *
from modules.models.T_baseline_model import *
from modules.tokenizers.base_tokenizer import *

WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")
TOKEN_MODEL_NAME = 'token_model_baseline'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

RuntimeError: Parent directory C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\token_model_baseline does not exist.

WORKDIR: C:\Users\Roman\Documents\Projects\code_autocomplete
[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1


## Training loop

In [5]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_token_model(
    model:           TokenModel,
    train_dl:        DataLoader,
    val_dl:          DataLoader,
    epochs:          int,
    lr:              float,
    device:          torch.device,
    saver:           BestModelSaver,
    log:             MetricLog,
    plot_dir:        str,
    label_smoothing: float = 0.1,
    warmup_frac:     float = 0.05,
    patience:        int   = 3,
    use_amp:         bool  = True,
):
    tqdm.write(f"[Token] DataLoader — {len(train_dl)} train batches, "
               f"{len(val_dl)} val batches")

    # ── optimiser + warmup-cosine schedule ───────────────────────────────────
    opt          = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps  = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    # ── label smoothing built into CE ───────────────────────────────────────
    crit = nn.CrossEntropyLoss(
        ignore_index=SPECIAL["<PAD>"],
        label_smoothing=label_smoothing,
    )

    # ── mixed precision ─────────────────────────────────────────────────────
    amp_enabled = use_amp and device.type == "cuda"
    scaler      = GradScaler("cuda", enabled=amp_enabled)

    # ── early stopping ──────────────────────────────────────────────────────
    stopper = EarlyStopping(patience=patience)

    for ep in range(1, epochs + 1):
        # ── train ────────────────────────────────────────────
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0
        gn = 0.0

        for x, y in tqdm(train_dl, desc=f"[Token] Epoch {ep}/{epochs} train",
                         leave=False, unit="batch"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(x)
                loss   = crit(logits.view(-1, logits.size(-1)), y.view(-1))

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gn = _clip_norm(model)
            scaler.step(opt)
            scaler.update()
            sched.step()                       # per-batch step

            with torch.no_grad():
                preds = logits.argmax(-1)
                mask  = (y != SPECIAL["<PAD>"])
                t_acc += (preds[mask] == y[mask]).float().mean().item()
            t_loss  += loss.item()
            t_steps += 1

        tl = t_loss / t_steps
        ta = t_acc  / t_steps

        # ── val ──────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for x, y in tqdm(val_dl, desc=f"[Token] Epoch {ep}/{epochs} val  ",
                             leave=False, unit="batch"):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                    logits = model(x)
                    loss   = crit(logits.view(-1, logits.size(-1)), y.view(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=ta, grad_norm=gn)
        tqdm.write(
            f"[Token ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{TOKEN_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{TOKEN_MODEL_NAME}_ep{ep:02d}.png")

        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break

    plot_metrics(log, f"{TOKEN_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{TOKEN_MODEL_NAME}_final.png")

## Main

In [6]:
class Arguments():
    def __init__(self, data_dir: str = f"{WORKDIR}/Clean_Dataset", ckpt_dir: str = f"{WORKDIR}/checkpoints/{TOKEN_MODEL_NAME}",
                    plot_dir: str = F"{WORKDIR}/plots/{TOKEN_MODEL_NAME}", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    args = Arguments(epochs=2, max_files=5)
    # args = Arguments(max_files=100)
    # args = Arguments(skip_line=True, max_files=100, epochs=1)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    # args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads,  n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )

    torch.serialization.add_safe_globals([ModelCfg])

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        tm = TokenModel(cfg).to(device)
        tok_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / TOKEN_MODEL_NAME + "_*.pt")))
        if tok_paths:
            ck = torch.load(tok_paths[0], map_location=device, weights_only=False)
            tm.load_state_dict(ck["model_state"])
            print(f"[Loaded] token model from {tok_paths[0]}")
    
        hand_test_repl(tm, None, tokenizer, None, device)
        return
    

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]
    
    # ── TOKEN MODEL ──────────────────────────────────────────
    if not args.skip_token:
        print("  Prepairing TOKEN model")

        all_ids_tr = []
        for t in tr_txt:
            all_ids_tr.extend(tokenizer.encode(t))
        all_ids_va = []
        for t in va_txt:
            all_ids_va.extend(tokenizer.encode(t))

        tr_ds = TokenDataset(all_ids_tr, args.ctx)
        va_ds = TokenDataset(all_ids_va, args.ctx)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,  num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False, num_workers=0, pin_memory=True)

        tok_model = TokenModel(cfg).to(device)
        n_params  = sum(p.numel() for p in tok_model.parameters() if p.requires_grad)
        print(f"[Token Model] {n_params/1e6:.2f}M parameters")

        tok_saver = BestModelSaver(args.ckpt_dir, TOKEN_MODEL_NAME)
        tok_log   = MetricLog()
        print("  Training TOKEN model")
        train_token_model(tok_model, tr_dl, va_dl, args.epochs, args.lr,
                          device, tok_saver, tok_log, args.plot_dir)

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(tok_model, None, tokenizer, None, device)


main()

[Tokenizer] loading tokenizer.json
[Loading] Started loading
[Data] loaded 5 files from C:\Users\Roman\Documents\Projects\code_autocomplete/Clean_Dataset
[Loading] Ended loading
  Prepairing TOKEN model
[Token Model] 3.16M parameters
  Training TOKEN model
[Token] DataLoader — 767 train batches, 599 val batches
Epoch 1


[Token ep   1] train_loss=0.3584  val_loss=0.3488  ppl=1.4  acc=0.995  lr=2.70e-04
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\token_model_baseline\token_model_baseline_ep001_loss0.3488.pt  (val_loss=0.3488)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/token_model_baseline/token_model_baseline_ep01.png
Epoch 2


[Token ep   2] train_loss=0.3497  val_loss=0.3489  ppl=1.4  acc=1.000  lr=0.00e+00
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\token_model_baseline\token_model_baseline_ep002_loss0.3489.pt  (val_loss=0.3489)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/token_model_baseline/token_model_baseline_ep02.png
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/token_model_baseline/token_model_baseline_final.png
